# Publication-safe notebook

This generalized reference implementation expects a live SparkSession named
`spark` and pseudonymous tables matching `docs/data-contract.md`. It contains no
institution-specific ingestion, credentials, internal paths, data, or results.

Parameter defaults are illustrative and intentionally differ from internal
operating settings. Calibrate them only with authorized development data.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import StorageLevel
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import IPython.display as _disp

import os

WRITE_DB = os.environ.get("FRAUD_GRAPH_WRITE_DB", "fraud_graph_demo")
EMP = os.environ.get("FRAUD_GRAPH_PREFIX", "onboarding_fraud")
SCORE_TBL = f"{WRITE_DB}.{EMP}_v2_final_scores"
FEATS_TBL = f"{WRITE_DB}.{EMP}_v2_final_features"
EDGES_TXN = f"{WRITE_DB}.{EMP}_v2_edges_transferred"
OWN_TBL   = f"{WRITE_DB}.{EMP}_v2_edges_ownership"
SUBG_NIDX = f"{WRITE_DB}.{EMP}_v2_subgraph_node_index"

ANCHOR_TRAILING = {
    "202401": ["202310", "202311", "202312"],
    "202402": ["202311", "202312", "202401"],
    "202403": ["202312", "202401", "202402"],
}
INVESTIGATE_ANCHOR = "202401"
TOP_N = int(os.environ.get("FRAUD_GRAPH_INVESTIGATION_TOP_N", "25"))
N_VIZ = int(os.environ.get("FRAUD_GRAPH_INVESTIGATION_N_VIZ", "5"))
RARE_BENEF_MAX = int(os.environ.get("FRAUD_GRAPH_RARE_BENEF_MAX", "25"))
MAX_MULES_VIZ = int(os.environ.get("FRAUD_GRAPH_MAX_MULES_VIZ", "5"))
MAX_SEEDS_PER_MULE = int(os.environ.get("FRAUD_GRAPH_MAX_SEEDS_PER_MULE", "2"))
MAX_FEEDERS_PER_MULE = int(os.environ.get("FRAUD_GRAPH_MAX_FEEDERS_PER_MULE", "5"))
MAX_COFEEDERS_VIZ = int(os.environ.get("FRAUD_GRAPH_MAX_COFEEDERS_VIZ", "8"))
REASON_FEATS = ["n_shared_mule", "closeness_to_bad", "n_shared_atm", "comm_bad_rate", "fraudar_score", "in_dense_block"]

OUT_QUEUE = f"{WRITE_DB}.{EMP}_v2_investigation_queue"

spark.conf.set("spark.sql.shuffle.partitions", os.environ.get("FRAUD_GRAPH_SHUFFLE_PARTITIONS", "200"))

def save_overwrite(df, t):
    spark.sql("DROP TABLE IF EXISTS " + t); df.write.mode("overwrite").saveAsTable(t)

print("investigation | anchor:", INVESTIGATE_ANCHOR, "| top-N queue:", TOP_N, "| rings to draw:", N_VIZ)


In [ ]:
feats_cols = spark.table(FEATS_TBL).columns
have = [c for c in REASON_FEATS if c in feats_cols]
scores = spark.table(SCORE_TBL).select("cust", "anchor_month", "target_fraud", "final_score", "risk_pctl")
feats  = spark.table(FEATS_TBL).select("cust", "anchor_month", *have)
qa = (scores.where(F.col("anchor_month") == INVESTIGATE_ANCHOR)
      .join(feats, ["cust", "anchor_month"], "left"))
w = Window.partitionBy("anchor_month").orderBy(F.col("risk_pctl").desc_nulls_last())
top = qa.withColumn("rank", F.row_number().over(w)).where(F.col("rank") <= TOP_N)
tp = top.toPandas()

def reasons(r):
    out = []
    if float(r.get("n_shared_mule", 0) or 0) > 0:
        out.append("shares %d rare mule(s) with known fraud" % int(r["n_shared_mule"]))
    if float(r.get("n_shared_atm", 0) or 0) > 0:
        out.append("shares %d cash-out ATM(s) with known fraud" % int(r["n_shared_atm"]))
    cl = float(r.get("closeness_to_bad", 0) or 0)
    if cl > 0:
        out.append("close to known fraud in the transfer graph (closeness %.2f)" % cl)
    if float(r.get("in_dense_block", 0) or 0) > 0:
        out.append("member of a dense Fraudar-style block (diagnostic flag)")
    if float(r.get("comm_bad_rate", 0) or 0) > 0:
        out.append("in a community with elevated fraud density (%.3f)" % float(r["comm_bad_rate"]))
    if float(r.get("fraudar_score", 0) or 0) > 0:
        out.append("elevated Fraudar suspiciousness score")
    return " | ".join(out) if out else "no graph signal (ranked low)"

tp["why_flagged"] = tp.apply(reasons, axis=1)
QUEUE_COLS = ["anchor_month", "rank", "cust", "risk_pctl", "final_score", "target_fraud", "why_flagged"] + have
qout = spark.createDataFrame(tp[QUEUE_COLS])
save_overwrite(qout, OUT_QUEUE)
print("saved review queue:", OUT_QUEUE, "| rows:", len(tp))
_disp.display(tp[["rank", "cust", "risk_pctl", "target_fraud", "why_flagged"]].head(20))


In [ ]:
n_top = len(tp); n_fraud = int(tp["target_fraud"].sum())
print("=== top-%d review queue @ %s ===" % (n_top, INVESTIGATE_ANCHOR))
print("   precision of the top-%d = %d fraud / %d reviewed = %.1f%%"
      % (n_top, n_fraud, n_top, 100.0 * n_fraud / max(n_top, 1)))
for c in have:
    fired = int((tp[c].astype(float) > 0).sum())
    fired_bad = int(((tp[c].astype(float) > 0) & (tp["target_fraud"] == 1)).sum())
    print("   signal %-18s fired on %3d/%d queued  (of which %d are fraud)" % (c, fired, n_top, fired_bad))


In [ ]:
months = ANCHOR_TRAILING[INVESTIGATE_ANCHOR]
e0 = (spark.table(EDGES_TXN).where(F.col("txn_month").isin(*months))
      .select("src", "dst", F.col("sum_amount").cast("double").alias("amt"))
      .where(F.col("src") != F.col("dst")))
fanin = e0.groupBy("dst").agg(F.countDistinct("src").alias("fi"))
rare = fanin.where(F.col("fi") <= RARE_BENEF_MAX).select("dst")
e = (e0.join(rare, "dst", "inner").groupBy("src", "dst").agg(F.sum("amt").alias("amt"))
     .persist(StorageLevel.DISK_ONLY))
e.count()
lab = spark.table(SUBG_NIDX).where(F.col("anchor_month") == INVESTIGATE_ANCHOR)
seeds = lab.where(F.col("is_known_bad") == 1).select(F.col("id").alias("seed_acc")).distinct()
seed_mules = (e.alias("e").join(seeds.alias("s"), F.col("e.src") == F.col("s.seed_acc"))
              .select(F.col("e.dst").alias("mule")).distinct())

_gc = tp[tp.get("n_shared_mule", pd.Series([0] * len(tp))).astype(float) > 0].copy()
viz = _gc.sort_values(["n_shared_mule", "risk_pctl"], ascending=[False, False]).head(N_VIZ).reset_index(drop=True)
viz_custs = viz["cust"].tolist()
print("drawing the %d densest rings among queued graph-connected customers:" % len(viz_custs),
      [(str(c)[-6:], int(m)) for c, m in zip(viz_custs, viz["n_shared_mule"])])

cdf = pd.DataFrame(columns=["cust", "cand_acct", "mule", "amt"])
sdf = pd.DataFrame(columns=["seed_acct", "mule"])
fdf = pd.DataFrame(columns=["feeder", "mule"])
if viz_custs:
    own = spark.table(OWN_TBL).select(F.col("src").alias("cust"), F.col("dst").alias("acct"))
    viz_accts = own.where(F.col("cust").isin(viz_custs))
    cand_e = (e.alias("e").join(viz_accts.alias("v"), F.col("e.src") == F.col("v.acct"))
              .join(seed_mules.alias("m"), F.col("e.dst") == F.col("m.mule"))
              .select(F.col("v.cust").alias("cust"), F.col("e.src").alias("cand_acct"),
                      F.col("e.dst").alias("mule"), F.col("e.amt").alias("amt")).distinct())
    cand_rows = cand_e.collect()
    cdf = pd.DataFrame([(r["cust"], r["cand_acct"], r["mule"], float(r["amt"])) for r in cand_rows],
                       columns=["cust", "cand_acct", "mule", "amt"])
    if len(cdf):
        mules = cand_e.select("mule").distinct()
        seed_e = (e.alias("e").join(seeds.alias("s"), F.col("e.src") == F.col("s.seed_acc"))
                  .join(mules.alias("m"), F.col("e.dst") == F.col("m.mule"))
                  .select(F.col("e.src").alias("seed_acct"), F.col("e.dst").alias("mule")).distinct())
        sdf = pd.DataFrame([(r["seed_acct"], r["mule"]) for r in seed_e.collect()], columns=["seed_acct", "mule"])

        wf = Window.partitionBy("mule").orderBy(F.col("feeder"))
        feeders = (e.alias("e").join(mules.alias("m"), F.col("e.dst") == F.col("m.mule"))
                   .select(F.col("e.src").alias("feeder"), F.col("e.dst").alias("mule")))
        feeders_c = feeders.withColumn("rn", F.row_number().over(wf)).where(F.col("rn") <= MAX_FEEDERS_PER_MULE).drop("rn")
        fdf = pd.DataFrame([(r["feeder"], r["mule"]) for r in feeders_c.collect()], columns=["feeder", "mule"])
print("collected ring edges: candidate->mule =", len(cdf), "| seed->mule =", len(sdf), "| all feeders =", len(fdf))


In [ ]:
def _short(x):
    return str(x)[-5:]

def _ypos(n):
    return np.linspace(0.03, 0.97, n) if n > 1 else np.array([0.5])

drew = False
if len(viz) > 0 and len(cdf) > 0:
    ncol = 3; nrow = int(np.ceil(len(viz) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.6 * ncol, 4.6 * nrow))
    axes = np.array(axes).reshape(-1)
    for i in range(len(viz)):
        ax = axes[i]; vr = viz.iloc[i]; C = vr["cust"]
        cm = cdf[cdf.cust == C]
        mset = list(dict.fromkeys(cm["mule"].tolist()))[:MAX_MULES_VIZ]
        cm = cm[cm.mule.isin(mset)]
        cand_accts = list(dict.fromkeys(cm["cand_acct"].tolist()))
        sm2 = []
        for m in mset:
            for s in sdf[sdf.mule == m]["seed_acct"].tolist()[:MAX_SEEDS_PER_MULE]:
                sm2.append((s, m))
        seed_accts = list(dict.fromkeys([s for s, _ in sm2]))

        exclude = set(cand_accts) | set(seed_accts)
        co_e = fdf[fdf.mule.isin(mset) & (~fdf.feeder.isin(exclude))]
        co_nodes = list(dict.fromkeys(co_e["feeder"].tolist()))[:MAX_COFEEDERS_VIZ]
        co_e = co_e[co_e.feeder.isin(co_nodes)]
        left = [("c", a) for a in cand_accts] + [("o", a) for a in co_nodes]
        yl = _ypos(len(left)); ym = _ypos(len(mset)); ys = _ypos(len(seed_accts))
        pos = {}
        for k, (t, a) in enumerate(left):  pos[(t, a)] = (0.0, yl[k])
        for k, m in enumerate(mset):       pos[("m", m)] = (1.0, ym[k])
        for k, s in enumerate(seed_accts): pos[("s", s)] = (2.0, ys[k])
        amax = float(cm["amt"].max()) if len(cm) and cm["amt"].max() > 0 else 1.0
        for _, er in cm.iterrows():
            x1, y1 = pos[("c", er["cand_acct"])]; x2, y2 = pos[("m", er["mule"])]
            ax.plot([x1, x2], [y1, y2], color="#c0392b", lw=0.8 + 2.6 * (float(er["amt"]) / amax), alpha=0.8, zorder=2)
        for _, er in co_e.iterrows():
            if ("o", er["feeder"]) not in pos or ("m", er["mule"]) not in pos: continue
            x1, y1 = pos[("o", er["feeder"])]; x2, y2 = pos[("m", er["mule"])]
            ax.plot([x1, x2], [y1, y2], color="#b0b3b8", lw=0.7, alpha=0.55, zorder=1)
        for s, m in sm2:
            x1, y1 = pos[("s", s)]; x2, y2 = pos[("m", m)]
            ax.plot([x1, x2], [y1, y2], color="#222222", lw=1.0, alpha=0.7, ls="--", zorder=2)
        style = {"c": ("#d62728", "o", 150), "o": ("#9aa0a6", "o", 55), "m": ("#ff8c1a", "s", 150), "s": ("#111111", "^", 150)}
        for (typ, a), (x, y) in pos.items():
            col, mk, sz = style[typ]
            ax.scatter([x], [y], c=col, marker=mk, s=sz, zorder=3, edgecolors="white", linewidths=0.6)
            if typ in ("c", "m", "s"):
                ax.annotate(_short(a), (x, y), fontsize=6, ha="center", va="center", color="white", zorder=4)
        outcome = "FRAUD" if int(vr["target_fraud"]) == 1 else "not (yet)"
        ax.set_title("cust ...%s | risk %.1f | %d mule(s), %d seed(s), %d co-feeder(s) | %s"
                     % (_short(C), float(vr["risk_pctl"]), len(mset), len(seed_accts), len(co_nodes), outcome), fontsize=8)
        ax.set_xlim(-0.45, 2.45); ax.set_ylim(-0.06, 1.06); ax.axis("off")
        ax.text(0.0, -0.03, "candidate + co-feeders", ha="center", fontsize=6.5, color="#555555")
        ax.text(1.0, -0.03, "shared mule", ha="center", fontsize=6.5, color="#ff8c1a")
        ax.text(2.0, -0.03, "known fraud", ha="center", fontsize=6.5, color="#111111")
    for j in range(len(viz), len(axes)):
        axes[j].axis("off")
    fig.suptitle("Collector rings @ %s  (red = candidate, grey = co-feeder, orange = shared rare mule, black = known-fraud "
                 "seed;  red line width ~ amount, dashed = known fraud pays SAME mule)" % INVESTIGATE_ANCHOR, fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.97]); plt.show(); drew = True
if not drew:
    print("No graph-connected customers in the top-%d to draw (all top rows have n_shared_mule=0), or no ring edges "
          "collected. The rings only exist for the graph-connected slice; the rest are ranked by other signals." % TOP_N)

e.unpersist()


In [ ]:
try:
    import networkx as nx
    _HAS_NX = True
except Exception as _e:
    _HAS_NX = False
    print("networkx unavailable (%s) -> skipping the neighbourhood view; the columnar rings in D still cover it." % _e)

if _HAS_NX and len(viz) > 0 and len(cdf) > 0 and len(fdf) > 0:
    seed_set = set(sdf["seed_acct"].astype(str).tolist())
    ncol = 3; nrow = int(np.ceil(len(viz) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.4 * ncol, 4.6 * nrow)); axes = np.array(axes).reshape(-1)
    for i in range(len(viz)):
        ax = axes[i]; vr = viz.iloc[i]; C = vr["cust"]
        my_mules = list(dict.fromkeys(cdf[cdf.cust == C]["mule"].astype(str).tolist()))[:MAX_MULES_VIZ]
        my_cands = set(cdf[(cdf.cust == C) & (cdf.mule.astype(str).isin(my_mules))]["cand_acct"].astype(str).tolist())
        fe = fdf[fdf.mule.astype(str).isin(my_mules)]
        G = nx.DiGraph()
        for _, er in fe.iterrows():
            G.add_edge(str(er["feeder"]), str(er["mule"]))
        if G.number_of_nodes() == 0:
            ax.axis("off"); continue
        def _role(node, mm=my_mules, mc=my_cands, ss=seed_set):
            if node in mm: return "mule"
            if node in mc: return "cand"
            if node in ss: return "seed"
            return "cofeeder"
        cmap = {"cand": "#d62728", "mule": "#ff8c1a", "seed": "#111111", "cofeeder": "#9aa0a6"}
        roles = {n: _role(n) for n in G.nodes()}
        colors = [cmap[roles[n]] for n in G.nodes()]
        sizes = [220 + 90 * G.degree(n) for n in G.nodes()]
        pos = nx.spring_layout(G, k=0.9, seed=42, iterations=60)
        nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.25, arrows=False, width=0.8)
        nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=sizes, edgecolors="white", linewidths=0.6)
        lab = {n: str(n)[-4:] for n in G.nodes() if roles[n] in ("mule", "cand", "seed")}
        nx.draw_networkx_labels(G, pos, labels=lab, ax=ax, font_size=6)
        n_seed = sum(1 for v in roles.values() if v == "seed"); n_co = sum(1 for v in roles.values() if v == "cofeeder")
        ax.set_title("cust ...%s | risk %.1f | %d mules, %d seeds, %d other feeders | %s"
                     % (str(C)[-5:], float(vr["risk_pctl"]), len(my_mules), n_seed, n_co,
                        "FRAUD" if int(vr["target_fraud"]) == 1 else "not (yet)"), fontsize=8)
        ax.axis("off")
    for j in range(len(viz), len(axes)):
        axes[j].axis("off")
    fig.suptitle("Neighbourhood ego-networks @ %s  (spring layout; red = candidate, orange = shared mule, "
                 "black = known fraud, grey = other feeder; node size ~ degree)" % INVESTIGATE_ANCHOR, fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()
elif _HAS_NX:
    print("No graph-connected customers / feeder edges to render in the neighbourhood view (columnar rings in D still apply).")


In [ ]:
import pandas as _pd
_fe = spark.table(FEATS_TBL).where(F.col("anchor_month") == INVESTIGATE_ANCHOR).toPandas()
for _col in _fe.columns:
    if _col not in ("cust", "anchor_month"):
        _fe[_col] = _pd.to_numeric(_fe[_col], errors="coerce")
_base_rate = 100.0 * _fe["target_fraud"].fillna(0).astype(int).mean() if "target_fraud" in _fe.columns else 0.0

def _tail(x, n=6):
    return "..." + str(x)[-n:]

def _pick(col, prefer_fraud=True):

    if col not in _fe.columns:
        return None
    d = _fe[_fe[col].fillna(0) > 0]
    if not len(d):
        return None
    if prefer_fraud and "target_fraud" in d.columns and (d["target_fraud"] == 1).any():
        d = d[d["target_fraud"] == 1]
    return d.sort_values(col, ascending=False).iloc[0]

def _outcome(r):
    return "later CONFIRMED fraud" if int(r.get("target_fraud", 0) or 0) == 1 else "clean / not yet fraud"

print("=== WORKED EXAMPLES (anchor %s | base fraud rate %.2f%%) ===\n" % (INVESTIGATE_ANCHOR, _base_rate))

_r = _pick("n_shared_mule")
if _r is not None:
    _cst = _r["cust"]; _nm = int(_r["n_shared_mule"])
    _line = "1) SHARED MULE (n_shared_mule=%d) - customer %s pays %d private mule(s) that a known fraud also pays; %s." % (
        _nm, _tail(_cst), _nm, _outcome(_r))
    _cm = cdf[cdf.cust == _cst] if len(cdf) else cdf
    if len(_cm):
        _m = _cm.iloc[0]["mule"]; _amt = float(_cm.iloc[0]["amt"])
        _sd = sdf[sdf.mule == _m] if len(sdf) else sdf
        _sname = _tail(_sd.iloc[0]["seed_acct"]) if len(_sd) else "a known-fraud account"
        _line += "\n     e.g. acct %s -> mule %s (Rs %.0f); known fraud %s -> the SAME mule %s." % (
            _tail(_cm.iloc[0]["cand_acct"]), _tail(_m), _amt, _sname, _tail(_m))
    print(_line + "\n")

_r = _pick("closeness_to_bad")
if _r is not None:
    _hops = _r["min_hops_to_bad"] if "min_hops_to_bad" in _fe.columns and _pd.notna(_r.get("min_hops_to_bad")) else None
    _h = (" in %d hop(s)" % int(_hops)) if _hops is not None else ""
    print("2) CLOSE TO FRAUD (closeness_to_bad=%.2f) - customer %s reaches a known fraud%s along time-ordered money flow; %s.\n"
          % (float(_r["closeness_to_bad"]), _tail(_r["cust"]), _h, _outcome(_r)))

_r = _pick("out_deg")
if _r is not None:
    print("3) FAN-OUT (out_deg=%d) - customer %s sends money to %d distinct private accounts (dispersal / structuring-out); %s.\n"
          % (int(_r["out_deg"]), _tail(_r["cust"]), int(_r["out_deg"]), _outcome(_r)))

_r = _pick("passthrough_ratio")
if _r is not None:
    print("4) PASS-THROUGH (passthrough_ratio=%.2f) - customer %s's money-in ~ money-out (near 1 = flows straight through, a layering mule); %s.\n"
          % (float(_r["passthrough_ratio"]), _tail(_r["cust"]), _outcome(_r)))

_r = _pick("on_3cycle")
if _r is not None:
    _nc = int(_r.get("n_3cycle_as_a", 0) or 0)
    print("5) ROUND-TRIP CYCLE (on_3cycle=1%s) - customer %s sits on a 3-account money loop (A->B->C->A); %s.\n"
          % ((", originates %d" % _nc) if _nc else "", _tail(_r["cust"]), _outcome(_r)))

_r = _pick("in_dense_block")
if _r is None:
    _r = _pick("fraudar_score")
if _r is not None:
    _bd = float(_r.get("block_density", 0) or 0)
    print("6) DENSE FRAUD BLOCK (in_dense_block=%d, density=%.3f) - customer %s belongs to a tightly-coordinated ring found by Fraudar; %s.\n"
          % (int(_r.get("in_dense_block", 0) or 0), _bd, _tail(_r["cust"]), _outcome(_r)))

_r = _pick("n_shared_atm")
if _r is not None:
    print("7) SHARED CASH-OUT (n_shared_atm=%d) - customer %s cashes out at %d ATM/POS terminal(s) a known fraud also uses; %s.\n"
          % (int(_r["n_shared_atm"]), _tail(_r["cust"]), int(_r["n_shared_atm"]), _outcome(_r)))

print("Each card summarizes one pseudonymous evaluation row; export it only under an approved data-handling policy.")


In [ ]:
BIN_FEATURES = ["n_shared_mule", "closeness_to_bad", "out_deg", "passthrough_ratio", "fraudar_score"]
HOW = {
    "n_shared_mule":    "beneficiary self-join: count of private mules the customer pays that a known fraud also pays",
    "closeness_to_bad": "1/(1+hops) from a time-respecting money-flow BFS started at the known frauds",
    "out_deg":          "out-degree in the transfer graph (distinct private accounts paid)",
    "passthrough_ratio":"least(in,out)/greatest(in,out) of rupee flow through the account (layering)",
    "fraudar_score":    "density spread over the accounts x seed-mules bipartite (Fraudar suspiciousness)",
}

def _bin_series(x, max_bins=8):
    s = _pd.Series(np.asarray(x, float)); nu = int(s.nunique(dropna=True))
    if nu <= 1:
        return np.zeros(len(s), int)
    if nu <= max_bins:
        order = {v: i for i, v in enumerate(sorted(s.dropna().unique()))}
        return s.map(order).fillna(0).astype(int).values
    z = (s == 0); out = np.zeros(len(s), int)
    if z.mean() >= 0.02 and int((~z).sum()) >= max_bins:
        pos = ~z
        qr = _pd.qcut(s[pos].rank(method="first"), q=min(max_bins - 1, int((~z).sum())), labels=False, duplicates="drop")
        out[pos.values] = 1 + qr.astype(int).values; return out
    return _pd.qcut(s.rank(method="first"), q=max_bins, labels=False, duplicates="drop").astype(int).values

_yv = _fe["target_fraud"].fillna(0).astype(int).values if "target_fraud" in _fe.columns else np.zeros(len(_fe), int)
_totbad = int(_yv.sum())
print("=== BINNING tied to the examples (anchor %s | %d frauds) ===\n" % (INVESTIGATE_ANCHOR, _totbad))
for _c in BIN_FEATURES:
    if _c not in _fe.columns:
        continue
    _x = _fe[_c].fillna(0).astype(float).values
    _g = (_pd.DataFrame({"b": _bin_series(_x), "lo": _x, "hi": _x, "y": _yv})
          .groupby("b").agg(cnt=("y", "size"), bad=("y", "sum"), lo=("lo", "min"), hi=("hi", "max"))
          .sort_values("lo", ascending=False))
    _g["event_pct"] = (100.0 * _g["bad"] / _g["cnt"]).round(2)
    _g["cum_capt_pct"] = (100.0 * _g["bad"].cumsum() / max(_totbad, 1)).round(1)
    _ex = _pick(_c); _exval = float(_ex[_c]) if _ex is not None else None
    print("-- %-17s (built as: %s) --" % (_c, HOW.get(_c, "")))
    for _, _rr in _g.reset_index().iterrows():
        _rng = ("%.3g" % _rr["lo"]) if _rr["lo"] == _rr["hi"] else ("%.3g..%.3g" % (_rr["lo"], _rr["hi"]))
        _mark = "   <== example customer here" if (_exval is not None and _rr["lo"] - 1e-9 <= _exval <= _rr["hi"] + 1e-9) else ""
        print("   value %-16s | cust %6d | fraud %4d | rate %6.2f%% | cum-capt %5.1f%%%s"
              % (_rng, int(_rr["cnt"]), int(_rr["bad"]), _rr["event_pct"], _rr["cum_capt_pct"], _mark))
    if _exval is not None:
        print("   -> worked example %s has %s=%.3g (marked band above).\n" % (_tail(_ex["cust"]), _c, _exval))
    else:
        print("")
print("READ: riskiest value band first; 'rate' = fraud rate inside that band; 'cum-capt' = share of ALL fraud caught down to here.")
print("Read it as: build the feature, bin the values, compare event rates, and mark the worked pseudonymous example.")


In [ ]:
print("=== Notebook 08 (v2) summary ===")
print("review queue :", OUT_QUEUE, "| rows:", spark.table(OUT_QUEUE).count(), "| anchor:", INVESTIGATE_ANCHOR)
print("Each queued customer carries a `why_flagged` reason (which signals fired) for the analyst.")
print("Two graph views of the top graph-connected customers:")
print("   D = full columnar ring: candidate + co-feeders -> shared RARE mule <- known-fraud seed (edge width ~ amount).")
print("   E = networkx spring-layout neighbourhood (organic view of the same ring).")
print("   Drawn customers = the DENSEST rings among the queued graph-connected set (not just top-by-risk).")
print("Example workflow: review the queue by risk_pctl; use the graph and reason codes as case context;")
print("route configured diagnostic flags for review under the approved policy.")
print("NOTEBOOK 08 (v2) COMPLETE — investigation output + graph visualisation.")
print("To investigate another cohort, set INVESTIGATE_ANCHOR to 202402 / 202403 and re-run.")
